# Notebook 3: Model Training & Time-Series Cross-Validation

## Theoretical Foundation
According to **Section 2.5.5 (Cross-validation for time-series data)**, traditional train/test splits are insufficient for time series. We implement **Rolling Origin Evaluation** (Time-Series CV) to rigorously test our models.
Furthermore, we employ powerful gradient boosting algorithms (LightGBM) which handle non-linear relationships well (Section 2.7.10).

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

df = pd.read_parquet('data/02_hierarchical_features.parquet')
df['month'] = pd.to_datetime(df['month'])
df.sort_values(['series_id', 'month'], inplace=True)

FEATURE_COLS = [c for c in df.columns if c.startswith(('lag', 'roll', 'month_', 'quarter'))]
TARGET = 'demand_transformed'

print(f"Features: {FEATURE_COLS}")

### 1. Rolling Origin Time-Series Cross Validation
We train the model over multiple expanding windows, evaluating on the immediate next month (H=1) up to H=6.

In [ ]:
# Define rolling origin cutoffs
months = sorted(df['month'].unique())
cv_splits = []
test_horizon = 6
# We take the last 3 possible origins for CV
for i in range(1, 4):
    test_start_idx = -test_horizon * i
    train_end = months[test_start_idx - 1]
    test_start = months[test_start_idx]
    test_end = months[test_start_idx + test_horizon - 1]
    cv_splits.append((train_end, test_start, test_end))

print("Time Series CV Splits:")
for i, (tr_e, te_s, te_e) in enumerate(cv_splits):
    print(f"Fold {i+1}: Train until {tr_e.date()} | Test from {te_s.date()} to {te_e.date()}")

### 2. Train and Evaluate Base Models
We train a single global LightGBM model across all hierarchical levels. Global models often outperform local models by learning cross-series patterns (Section 2.7.1).

In [ ]:
fold_results = []
models = []

for i, (train_end, test_start, test_end) in enumerate(cv_splits):
    train_mask = df['month'] <= train_end
    test_mask = (df['month'] >= test_start) & (df['month'] <= test_end)
    
    X_train, y_train = df[train_mask][FEATURE_COLS], df[train_mask][TARGET]
    X_test, y_test = df[test_mask][FEATURE_COLS], df[test_mask][TARGET]
    
    # Train model
    model = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.05, random_state=42, verbose=-1)
    model.fit(X_train, y_train)
    models.append(model)
    
    # Predict
    preds = model.predict(X_test)
    
    # Evaluate (on transformed scale for now)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)
    
    fold_results.append({'Fold': i+1, 'RMSE': rmse, 'MAE': mae})
    print(f"Fold {i+1} -> RMSE: {rmse:.3f}, MAE: {mae:.3f}")

cv_df = pd.DataFrame(fold_results)
print("\nAverage CV Performance:")
print(cv_df.mean())

# Save the model from the most recent fold
import joblib
joblib.dump(models[0], 'data/03_lgbm_model.pkl')